# 07 — The Operator Sinkhorn Iteration

In `04/06` a black-box convex solver handed us the entropic quantum optimal-transport plan. Here we build the explicit **operator Sinkhorn iteration** that *reaches* that same optimum on its own — alternating positive-operator rescalings of the Gibbs kernel, the matrix lift of the row/column rescaling that made classical transport fast in `03/10`.

**Prerequisites:** `04/06_entropic_qot_gibbs`, `03/10_sinkhorn_algorithm`.

**What you'll be able to do**
- Explain why an explicit **fixed-point iteration** is worth having even when a convex solver already returns the answer.
- Run `operator_sinkhorn` on a small quantum coupling problem and watch its marginal residual fall **geometrically** on a log scale.
- Cross-check the iteration against the `04/06` SDP in the commuting case and read the agreement as evidence the algorithm is real, not a wrapper.
- Turn the $\varepsilon$ dial and watch the plan move **sharp $\leftrightarrow$ blurry**, from a concentrated near-permutation coupling toward the product $\rho_A \otimes \rho_B$.

Welcome back. You arrive holding the entropic problem and the Gibbs kernel at its centre — and a small debt. `04/06` *described* the regularised optimum and then let cvxpy compute it, the same way `03/09` described the classical entropic plan before `03/10` showed you the five-line iteration that built it. Today we pay that debt in the quantum world. We write the operator analogue of Sinkhorn's alternating rescaling, run it, and watch it walk to the very plan the solver returned. By the end you will have *run the real algorithm*, not invoked a solver — and you will have felt, in your own iterates, why this whole construction is called Sinkhorn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum.composite import partial_trace, tensor
from qot_course.quantum_ot.sdp import quadratic_position_cost
from qot_course.quantum_ot.sinkhorn import (
    matrix_gibbs_kernel,
    operator_sinkhorn,
    quantum_sinkhorn_sdp,
)

np.random.seed(42)
viz.use_course_style()

## From the SDP to an iteration

In `04/06` we wrote the entropic quantum OT problem

$$
\rho^\star_{AB,\,\varepsilon} \,=\, \arg\min_{\rho_{AB}\,\in\,\Pi(\rho_A, \rho_B)}\;
       \mathrm{tr}(C\,\rho_{AB}) \;-\; \varepsilon\, S(\rho_{AB}),
$$

handed it to cvxpy's `von_neumann_entr` atom, and read the plan it returned. That is the right move when you want a trustworthy reference value — but it leaves the *mechanism* hidden inside the solver. The classical story did not stop there, and neither should this one.

Recall `03/10`. The optimality conditions of the classical entropic problem pin the plan to the shape $P^\star_\varepsilon = \mathrm{diag}(u)\,K\,\mathrm{diag}(v)$, where $K_{ij} = e^{-C_{ij}/\varepsilon}$ is the Gibbs kernel and the positive vectors $u, v$ are exactly what makes the row and column sums equal the marginals $a$ and $b$. Sinkhorn's insight was that you never need to solve for $u$ and $v$ at once: alternately rescale the rows to fix one marginal, then the columns to fix the other, and the iteration converges geometrically to the unique fixed point. Five lines, two matrix–vector products per step.

The quantum lift keeps the skeleton and changes the actors. The plan takes the shape

$$
P \,=\, (X \otimes Y)\; K\; (X \otimes Y), \qquad K = \exp(-C/\varepsilon),
$$

where the diagonal scaling vectors $u, v$ become **positive operators** $X$ and $Y$, and the entrywise kernel becomes the matrix-exponential Gibbs kernel of `04/06`. One half-step freezes $Y$ and solves for the $X$ that forces the partial trace $\mathrm{tr}_B(P)$ to equal $\rho_A$; the next half-step freezes $X$ and solves for the $Y$ that forces $\mathrm{tr}_A(P) = \rho_B$. Each half-step is a closed-form positive rescaling (this is *operator scaling* in the sense of Gurvits, 2004), and in the commuting/diagonal case it collapses back to the elementwise $X_{ii} = \sqrt{\rho_{A,ii} / M_{ii}}$ of classical Sinkhorn.

Why bother, when `04/06` already gave us the answer? Three reasons, the same ones that justified `03/10`:

- **The mechanism is the lesson.** Seeing the plan *assembled* by repeated rescaling is what turns "the entropic optimum exists" into "here is how it is computed". A solver hides exactly the structure we want to teach.
- **It scales past the SDP's comfort zone.** Interior-point methods on the full $d_A d_B \times d_A d_B$ Hermitian variable grow expensive quickly; the iteration only ever touches the small per-subsystem operators $X$ and $Y$.
- **It names the geometry.** This iteration is the operator analogue of Sinkhorn's *iterative Bregman projection* — the thread `04/08` will pull on to reveal the relative-entropy projection hiding underneath. We need the algorithm in hand before we can interpret it.

The routine `operator_sinkhorn` implements this for us, returning the plan and a `log` whose `marginal_errors` records, sweep by sweep, how far the iterate's partial traces still sit from the targets.

## The operator Sinkhorn iteration

We run it on the same example `04/06` used for its entropic solve: the quadratic position cost on three positions $\{0, 1, 2\}$, with two commuting diagonal "position" states $\rho_A = \mathrm{diag}(0.5, 0.3, 0.2)$ and $\rho_B = \mathrm{diag}(0.2, 0.3, 0.5)$.

The choice of states is deliberate, and `04/06` told us why. Pairing a pure marginal against the maximally mixed $I/2$ would be a trap here: by the Araki–Lieb bound a pure state against $I/2$ pins the joint entropy to a single value for *every* coupling, so the entropy bonus is constant over the feasible set, the plan is $\varepsilon$-independent, and the operator scaling has nothing to do. These two diagonal states with room to spread have no such degeneracy — the $\varepsilon$ dial genuinely moves the plan, as §4 will show. We start at a moderate $\varepsilon = 0.5$ and read the convergence `log`.

In [ ]:
# Same setup as 04/06: quadratic position cost on positions {0, 1, 2}, two commuting
# diagonal states with room to spread (no Araki-Lieb pinning -- see the markdown above).
C = quadratic_position_cost([0.0, 1.0, 2.0])
rho_a = np.diag([0.5, 0.3, 0.2]).astype(complex)
rho_b = np.diag([0.2, 0.3, 0.5]).astype(complex)

epsilon = 0.5
plan_iter, log = operator_sinkhorn(rho_a, rho_b, C, epsilon=epsilon)

print(f"operator Sinkhorn at eps = {epsilon}")
print(f"  sweeps to converge        n_iter = {log['n_iter']}")
print(f"  final marginal residual          = {log['marginal_errors'][-1]:.2e}  (Frobenius)")
print()

# An honest coupling: the iterate's partial traces must be the prescribed marginals.
tr_b = partial_trace(plan_iter, keep=[0], dims=[3, 3])
tr_a = partial_trace(plan_iter, keep=[1], dims=[3, 3])
print(f"marginal A recovered (diag, want 0.5, 0.3, 0.2):  {np.round(np.diag(tr_b).real, 4)}")
print(f"marginal B recovered (diag, want 0.2, 0.3, 0.5):  {np.round(np.diag(tr_a).real, 4)}")
print()

# Commuting diagonal inputs => the plan stays real (no coherence to create).
print(f"largest |imaginary part| of the plan:  {np.max(np.abs(plan_iter.imag)):.2e}")

**Read the output.** The iteration converges: starting from the unscaled kernel it takes a few dozen sweeps to drive the marginal residual down to the tolerance, at which point tracing out either subsystem returns the prescribed diagonal marginal to that same precision. The iterate is an honest quantum coupling, built not by a solver but by repeated rescaling. And its imaginary part is zero to machine precision — these commuting diagonal states carry no coherence, so the operator scaling produces a real plan, exactly the behaviour `04/06` saw for the same pair. Now we look at *how* the residual fell.

In [ ]:
errors = log["marginal_errors"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(
    range(1, len(errors) + 1),
    errors,
    color=viz.COLORS["quantum"],
    lw=2,
    marker="o",
    markersize=3,
)
ax.axhline(1e-12, color=viz.COLORS["highlight"], linestyle="--", alpha=0.8,
           label="convergence tolerance")
ax.set_xlabel("operator Sinkhorn sweep")
ax.set_ylabel(r"marginal residual  $\|\mathrm{tr}_B P - \rho_A\|_F + \|\mathrm{tr}_A P - \rho_B\|_F$")
ax.set_title(rf"Geometric convergence of operator Sinkhorn  ($\varepsilon = {epsilon}$)", pad=12)
ax.legend()
plt.show()

**Read the figure.** The residual is plotted on a logarithmic vertical axis against the sweep number. After the first few sweeps the points fall along a **straight line** — and a straight line on a log scale is **geometric (exponential) convergence**: every sweep shrinks the marginal error by a roughly constant factor, exactly as classical Sinkhorn did in `03/10`. The line drops cleanly to the dashed tolerance and stops, which is where the loop reports `n_iter`. This is the operator generalisation of Sinkhorn's matrix-scaling convergence: the rate is set by $\varepsilon$ (we will see in §4 that smaller $\varepsilon$ makes the line shallower — more sweeps to reach the same floor), but the *shape* is the same geometric descent that made the classical algorithm fast.

## Validation — the iteration equals the SDP

Here is the payoff. We have run an explicit fixed-point iteration; `04/06` ran a convex solver on the same data. If the algorithm is real, the two should return the **same plan** — and in the commuting case they provably do. When $\rho_A$, $\rho_B$ and $C$ all commute (as our diagonal example does), both the operator iteration and the von-Neumann SDP reduce to *classical* Sinkhorn on the shared eigenbasis, so they must agree.

We solve the entropic problem both ways at $\varepsilon = 0.3$ and compare the plans directly with the Frobenius norm $\|P_{\mathrm{iter}} - P_{\mathrm{sdp}}\|_F$. (You may see a `CLARABEL` solver warning from the SDP about an inaccurate solution; that is the interior-point method reporting it stopped near — not exactly at — its tightened tolerance, and it is exactly why we cross-check rather than trust either route blindly.)

In [ ]:
epsilon = 0.3

# Route 1: the explicit operator Sinkhorn iteration (this notebook).
plan_iter, log = operator_sinkhorn(rho_a, rho_b, C, epsilon=epsilon)

# Route 2: the black-box von-Neumann-entropy SDP (04/06), our ground-truth reference.
_, plan_sdp = quantum_sinkhorn_sdp(rho_a, rho_b, C, epsilon=epsilon)

gap = float(np.linalg.norm(plan_iter - plan_sdp))

print(f"at eps = {epsilon}")
print(f"  operator Sinkhorn:  n_iter = {log['n_iter']}, "
      f"final residual = {log['marginal_errors'][-1]:.2e}")
print()
print(f"  || P_iter - P_sdp ||_F = {gap:.2e}")
print(f"  the two agree to solver tolerance:  {bool(gap < 1e-6)}")

**Read the output.** The plan built by the explicit iteration and the plan returned by the convex solver coincide to solver tolerance — their Frobenius difference sits at roughly $10^{-7}$, the accuracy floor of the interior-point method we are comparing against, not a defect of either. This is the moment the algorithm earns its name. We did not call a solver and dress it up; we ran an alternating operator rescaling, by hand, and it walked to the very optimum cvxpy reaches by an entirely different route. In the commuting case the iteration and the SDP are the same answer — the operator Sinkhorn iteration is real.

One honest caveat before we go further. This exact agreement is a **commuting-case** statement: when $\rho_A$, $\rho_B$ and $C$ all share an eigenbasis, both routes collapse to classical Sinkhorn and must match. Off the commuting case the relationship is a genuine subtlety. The operator iteration always converges to a well-defined fixed point — the operator-scaling / quantum-Schrödinger-bridge solution of Georgiou & Pavon (2015) — but whether that fixed point coincides exactly with the von-Neumann-entropy SDP optimum is not settled in general (Cole, Lostaglio, Verma & Wilde, 2023). We claim the agreement we can prove, on the commuting pair, and no more.

## The $\varepsilon$ trade-off — sharp versus blurry

We met this trade-off classically in `03/10`: tiny $\varepsilon$ keeps the plan a sharp diagonal close to the unregularised optimum, while large $\varepsilon$ blurs it toward the independent coupling $a \otimes b$. The operator iteration inherits the same dial, now turning a quantum coupling between two limits — the sharp transport plan and the product state $\rho_A \otimes \rho_B$.

We run `operator_sinkhorn` on our commuting pair at three regularisation strengths — a small, a moderate, and a large $\varepsilon$ — and show each resulting plan with `viz.plot_density_matrix`. Because these states are diagonal, every plan is real and supported on the diagonal of the $9 \times 9$ density matrix; each diagonal entry is the mass on one ordered pair of positions $(i, j)$, with the row index running as $3i + j$. Watch that diagonal redistribute as $\varepsilon$ grows.

In [ ]:
epsilons = [0.3, 1.0, 8.0]
plans = [operator_sinkhorn(rho_a, rho_b, C, epsilon=eps)[0] for eps in epsilons]

# Distance from each entropic plan to the product (independent) coupling rho_A (x) rho_B,
# the large-eps limit. A shrinking distance is the "blurring toward product" made numeric.
product = tensor(rho_a, rho_b)
for eps, plan in zip(epsilons, plans):
    dist = float(np.linalg.norm(plan - product))
    print(f"eps = {eps:>4}:  || P_eps - rho_A (x) rho_B ||_F = {dist:.4f}")
print()
print(f"for reference, the unregularised separation "
      f"|| rho_A (x) rho_B ||_F = {float(np.linalg.norm(product)):.4f}")

In [ ]:
# One density-matrix figure per epsilon, shown in sequence (sharp -> blurry).
for eps, plan in zip(epsilons, plans):
    viz.plot_density_matrix(
        plan,
        title=rf"Operator Sinkhorn plan at $\varepsilon = {eps}$",
    )
    plt.show()

**Read the figures.** The three figures appear in sequence, one per $\varepsilon$; in each, the left heatmap is the real part of the plan and the right is the imaginary part, which stays flat at zero throughout — no coherence, exactly as in `04/06`. The story lives on the diagonal of the real parts, and it sweeps cleanly from sharp to blurry as you read down the sequence.

- At the **small** $\varepsilon = 0.3$ the plan is **concentrated**: almost all the mass sits on a short near-permutation set of pairs — $(0,0)$, $(0,1)$, $(1,1)$, $(2,2)$ — the cheapest routes under the squared-distance cost, with only a faint leak elsewhere. This is the operator echo of the thin bright diagonal `03/10` showed at small $\varepsilon$.
- At the **moderate** $\varepsilon = 1.0$ that backbone is still visible but the mass has begun to spread onto neighbouring pairs — the coupling is softening.
- At the **large** $\varepsilon = 8.0$ the diagonal has broadened into a smooth spread across many pairs, the entropy bonus now dominating the transport cost. The printed distances confirm what the eye reads: $\|P_\varepsilon - \rho_A \otimes \rho_B\|_F$ falls steadily as $\varepsilon$ grows, the plan drifting toward the product state in which $A$ and $B$ are independent and transport cost has stopped mattering.

This is the same regularisation trade-off you tuned classically, lifted to operators and produced here entirely by the explicit iteration — small $\varepsilon$ to keep the transport geometry sharp, large $\varepsilon$ to buy spread and fast, stable convergence.

## Your turn

**Warm-up.** Run `operator_sinkhorn` on the commuting pair above at two regularisation strengths — say $\varepsilon = 0.5$ and $\varepsilon = 2.0$ — and read `log["n_iter"]` from each. Which $\varepsilon$ needs more sweeps to reach the tolerance, and does that match the geometric-convergence picture, where a gentler (larger) regularisation gives a steeper log-scale line and so converges in fewer steps?

**Go further.** Take the iterate `plan_iter` you computed at $\varepsilon = 0.3$ in §3 and verify directly that it is a faithful coupling: trace out subsystem $B$ with `partial_trace(plan_iter, keep=[0], dims=[3, 3])`, trace out subsystem $A$ with `keep=[1]`, and check that the two reduced states match $\rho_A$ and $\rho_B$ to a tolerance such as $10^{-6}$. Why is it reassuring that the marginals come out right *without* ever having imposed them as explicit constraints, the way the SDP did?

**Challenge.** Predict the convergence behaviour as $\varepsilon \to 0$ before testing it. The Gibbs kernel $K = \exp(-C/\varepsilon)$ becomes increasingly ill-conditioned as $\varepsilon$ shrinks — its smallest eigenvalues collapse toward zero — so the operator rescaling has less and less room to work. Predict whether the iteration should need more or fewer sweeps as $\varepsilon$ falls, then run `operator_sinkhorn` at a small $\varepsilon$ (for instance $0.2$, with a generous `n_iter`) and read `log["n_iter"]` to confirm. What does this say about why practitioners reach for a log-domain implementation when they push $\varepsilon$ toward zero?

## What you built

- You wrote down, and then *ran*, the **operator Sinkhorn iteration** — alternating positive-operator rescalings $(X \otimes Y)\,K\,(X \otimes Y)$ of the matrix Gibbs kernel that drive the partial traces onto $\rho_A$ and $\rho_B$, the faithful matrix lift of the row/column rescaling from `03/10`.
- You watched its marginal residual fall **geometrically** on a log scale and read off the number of sweeps to convergence — the algorithm building a valid quantum coupling in front of you.
- You cross-checked the iterate against the `04/06` SDP in the commuting case and found them equal to solver tolerance, so you know the iteration is the real computation, not a wrapper around a solver.
- You turned the $\varepsilon$ dial and saw the plan move **sharp $\leftrightarrow$ blurry**, from a concentrated near-permutation coupling at small $\varepsilon$ toward the product $\rho_A \otimes \rho_B$ at large $\varepsilon$.

Sit with what happened here: you did not ask a solver for the entropic optimum, you *computed* it — by hand, by rescaling, by the operator generalisation of an idea that is barely five lines long in the classical world. The algorithm that made optimal transport fast now runs on density matrices, and you have run it.

## What's next

In `08_quantum_amari_bridge` we ask the question this iteration has been quietly raising: *why* does alternating rescaling land on the entropic optimum at all? The answer is that each sweep is a relative-entropy (Umegaki) projection onto a marginal constraint, so the operator Sinkhorn iteration you ran above is literally an iterative Bregman projection — the quantum closing of the information-geometry-meets-transport loop.

## References

- R. Sinkhorn, "A relationship between arbitrary positive matrices and doubly stochastic matrices", *Annals of Mathematical Statistics* **35**, 876–879 (1964). DOI:10.1214/aoms/1177703591
- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *Advances in Neural Information Processing Systems* **26** (2013). DOI:10.48550/arXiv.1306.0895
- G. Peyré & M. Cuturi, "Computational optimal transport", *Foundations and Trends in Machine Learning* **11**, 355–607 (2019). DOI:10.1561/2200000073
- L. Gurvits, "Classical complexity and quantum entanglement", *Journal of Computer and System Sciences* **69**, 448–484 (2004). DOI:10.1016/j.jcss.2004.06.003
- T. T. Georgiou & M. Pavon, "Positive contraction mappings for classical and quantum Schrödinger systems", *Journal of Mathematical Physics* **56**, 033301 (2015). DOI:10.1063/1.4915289
- S. Cole, M. Lostaglio, K. Verma & M. M. Wilde, "Quantum Wasserstein distance based on an optimization over separable states", *IEEE Transactions on Information Theory* **69**, 6657–6678 (2023). DOI:10.1109/TIT.2023.3287993
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091

**Previous:** `notebooks/04_QuantumOptimalTransport/06_entropic_qot_gibbs.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/08_quantum_amari_bridge.ipynb`